# Correlation Mapping:

### Correlation Matrix:

In [23]:
import pandas as pd
import numpy as np

# Constants from metrics doc
MIN_TICKERS = 5

FILE_PATH = "../../data/processed/gdelt_ohlcv_join.parquet"

df = pd.read_parquet(FILE_PATH)

# Check data sparsity 
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Date range:", df["price_date"].min(), "->", df["price_date"].max())
print("Unique ticker-days:", df[["ticker", "price_date"]].drop_duplicates().shape[0])

# Parse dates
df["price_date"] = pd.to_datetime(df["price_date"], utc=True)
df["article_date"] = pd.to_datetime(df["article_date"], utc=True)

# Ensure numeric
num_cols = [
    "sentiment_score",
    "next_open",
    "next_close"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Returns using metrics doc formula
# r_i,t = (close - open) / open

df["ret"] = (df["next_close"] - df["next_open"]) / df["next_open"]


# Aggregate sentiment to daily level (t) using article_date

daily_sentiment = (
    df.groupby("article_date")
      .agg(
          sentiment_score=("sentiment_score", "mean")
      )
      .reset_index()
)

# Sort and shift sentiment forward to align with next trading day (t+1)

daily_sentiment = daily_sentiment.sort_values("article_date")
daily_sentiment["price_date"] = daily_sentiment["article_date"].shift(-1)

# Compute cross-sectional metrics per day

returns = (
    df.groupby(["ticker", "price_date"])
      .agg(ret=("ret", "first"))
      .reset_index()
)

cs_stats = (
    returns.groupby("price_date")
    .agg(
        ret_cs_std=("ret", "std"),
        ret_mean=("ret", "mean"),
        n_tickers_returns=("ticker", "nunique"),
        ret_cs_mad=("ret", lambda x: np.mean(np.abs(x - np.mean(x))))
    )
    .reset_index()
)

# coverage ratio (7 tickers total)
cs_stats["coverage_ratio"] = cs_stats["n_tickers_returns"] / 7

# ticker coverage threshold
cs_stats = cs_stats[cs_stats["n_tickers_returns"] >= MIN_TICKERS]


# Merge sentiment with dispersion metrics

corr_df = daily_sentiment.merge(cs_stats, on="price_date")

corr_df = corr_df.dropna()


# Correlation matrices

cols = [
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean"
]

print("\nPearson Correlation Matrix\n")
print(corr_df[cols].corr(method="pearson"))

print("\nSpearman Correlation Matrix\n")
print(corr_df[cols].corr(method="spearman"))

Rows: 89405
Columns: ['seendate', 'url', 'title', 'language', 'domain', 'socialimage', 'company', 'ticker', 'date', 'sentiment_score', 'sentiment_confidence', 'sentiment_label', 'article_date', 'price_date', 'next_open', 'next_high', 'next_low', 'next_close', 'next_adj_close', 'next_volume']
Date range: 2024-02-09 00:00:00 -> 2026-02-24 00:00:00
Unique ticker-days: 1401

Pearson Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000    0.009613    0.013237 -0.113296
ret_cs_std              0.009613    1.000000    0.969297  0.290072
ret_cs_mad              0.013237    0.969297    1.000000  0.349902
ret_mean               -0.113296    0.290072    0.349902  1.000000

Spearman Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000    0.062393    0.071783 -0.071408
ret_cs_std              0.062393    1.000000    0.964991  0.149392
ret_cs_mad              0.071783

### Rolling Correlations:

In [24]:
corr_df = corr_df.sort_values("price_date")

windows = [5, 10, 30]

for w in windows:
    
    corr_df[f"roll_corr_sent_std_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_std"])
    )

    corr_df[f"roll_corr_sent_mad_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_mad"])
    )

    corr_df[f"roll_corr_sent_mean_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_mean"])
    )

print("\nRolling correlation preview:\n")
print(
    corr_df[
        [
            "price_date",
            "roll_corr_sent_std_5d",
            "roll_corr_sent_std_10d",
            "roll_corr_sent_std_30d",
        ]
    ].tail(10)
)

# Diagnostics for sparsity and rolling window coverage

print("\nNon-null rolling counts:")
print(corr_df.filter(like="roll_corr").count())

print("\nFinal daily rows after MIN_TICKERS filter:", len(corr_df))
print("\nFirst available dates:")
print(corr_df["price_date"].sort_values().head(10))


Rolling correlation preview:

                   price_date  roll_corr_sent_std_5d  roll_corr_sent_std_10d  \
110 2026-02-05 00:00:00+00:00              -0.556000               -0.636601   
111 2026-02-06 00:00:00+00:00              -0.529660               -0.661735   
112 2026-02-09 00:00:00+00:00              -0.532884               -0.707552   
113 2026-02-10 00:00:00+00:00              -0.804440               -0.732547   
114 2026-02-11 00:00:00+00:00              -0.863351               -0.734855   
115 2026-02-12 00:00:00+00:00              -0.662454               -0.660754   
116 2026-02-17 00:00:00+00:00              -0.580529               -0.665846   
117 2026-02-18 00:00:00+00:00               0.835631               -0.315378   
118 2026-02-19 00:00:00+00:00               0.230620               -0.482578   
119 2026-02-23 00:00:00+00:00               0.205285               -0.499852   

     roll_corr_sent_std_30d  
110               -0.366407  
111               -0.370247 

### Export Summary Tables 

In [25]:
summary_cols = [
    "price_date",
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean",
    "n_tickers_returns",
    "coverage_ratio",
    "roll_corr_sent_std_5d",
    "roll_corr_sent_std_10d",
    "roll_corr_sent_std_30d",
    "roll_corr_sent_mad_5d",
    "roll_corr_sent_mad_10d",
    "roll_corr_sent_mad_30d",
    "roll_corr_sent_mean_5d",
    "roll_corr_sent_mean_10d",
    "roll_corr_sent_mean_30d",
]

summary_df = corr_df[summary_cols].copy()

print("\nRows exported:", len(summary_df))
print("Date range:", summary_df["price_date"].min(), "->", summary_df["price_date"].max())

output_path = "correlation_summary.csv"
summary_df.to_csv(output_path, index=False)

print(f"Saved summary table to {output_path}")
print(summary_df.head())


Rows exported: 120
Date range: 2024-02-09 00:00:00+00:00 -> 2026-02-23 00:00:00+00:00
Saved summary table to correlation_summary.csv
                 price_date  sentiment_score  ret_cs_std  ret_cs_mad  \
0 2024-02-09 00:00:00+00:00         0.014184    0.011921    0.009208   
1 2024-02-26 00:00:00+00:00         0.002399    0.020499    0.011775   
2 2024-03-04 00:00:00+00:00        -0.060906    0.021471    0.014472   
3 2024-03-11 00:00:00+00:00        -0.042349    0.014154    0.010947   
4 2024-03-19 00:00:00+00:00        -0.013564    0.013228    0.009069   

   ret_mean  n_tickers_returns  coverage_ratio  roll_corr_sent_std_5d  \
0  0.011527                  7             1.0                    NaN   
1 -0.003908                  7             1.0                    NaN   
2 -0.009831                  7             1.0                    NaN   
3 -0.003249                  7             1.0                    NaN   
4  0.008883                  7             1.0              -0.43531

### Spasity Diagnostics

In [26]:
import pandas as pd
import numpy as np

print("==== RAW DATA CHECKS ====")
print("Total article rows:", len(df))
print("Unique tickers:", df["ticker"].nunique())
print("Raw date range:", df["price_date"].min(), "->", df["price_date"].max())
print("Unique ticker-days in raw joined data:",
      df[["ticker", "price_date"]].drop_duplicates().shape[0])

# 1) Check missing price fields needed for return calculation
price_cols = ["next_open", "next_close"]
print("\nMissingness in price fields:")
for c in price_cols:
    miss = df[c].isna().mean() if c in df.columns else np.nan
    print(f"{c}: {miss:.2%}")

# 2) Build a clean ticker-day price panel explicitly
price_panel = (
    df[["ticker", "price_date", "next_open", "next_close"]]
      .copy()
      .dropna(subset=["ticker", "price_date", "next_open", "next_close"])
      .drop_duplicates(["ticker", "price_date"])
)

price_panel["ret"] = (price_panel["next_close"] - price_panel["next_open"]) / price_panel["next_open"]

print("\n==== PRICE PANEL CHECKS ====")
print("Ticker-day rows after requiring valid open/close:",
      len(price_panel))
print("Unique ticker-days in price panel:",
      price_panel[["ticker", "price_date"]].drop_duplicates().shape[0])
print("Price panel date range:",
      price_panel["price_date"].min(), "->", price_panel["price_date"].max())

print("\nTicker-day counts by ticker (price panel):")
print(price_panel.groupby("ticker").size().sort_values(ascending=False))

# 3) How many tickers are available per day?
ticker_counts = (
    price_panel.groupby("price_date")["ticker"]
    .nunique()
    .rename("n_tickers")
    .reset_index()
)

print("\n==== TICKER COVERAGE PER DAY ====")
print("Days with any valid returns:", len(ticker_counts))
print("\nDistribution of ticker coverage per day:")
print(ticker_counts["n_tickers"].value_counts().sort_index())

print("\nSummary stats of daily ticker coverage:")
print(ticker_counts["n_tickers"].describe())

# 4) How many days survive each possible threshold?
print("\n==== DAYS SURVIVING COVERAGE THRESHOLDS ====")
for k in range(1, 8):
    surviving = (ticker_counts["n_tickers"] >= k).sum()
    print(f"MIN_TICKERS >= {k}: {surviving} days")

# 5) Identify the actual days being dropped by MIN_TICKERS = 5
MIN_TICKERS = 5

kept_days = set(ticker_counts.loc[ticker_counts["n_tickers"] >= MIN_TICKERS, "price_date"])
dropped_days = ticker_counts.loc[ticker_counts["n_tickers"] < MIN_TICKERS].copy()

print(f"\nDays kept with MIN_TICKERS={MIN_TICKERS}: {(ticker_counts['n_tickers'] >= MIN_TICKERS).sum()}")
print(f"Days dropped with MIN_TICKERS={MIN_TICKERS}: {(ticker_counts['n_tickers'] < MIN_TICKERS).sum()}")

print("\nFirst 15 dropped days and their ticker counts:")
print(dropped_days.head(15))

# 6) Which tickers are missing most often on dropped days?
print("\n==== WHICH TICKERS ARE MISSING ON DROPPED DAYS? ====")

all_tickers = sorted(price_panel["ticker"].dropna().unique().tolist())

missing_records = []
for d in dropped_days["price_date"]:
    present = set(price_panel.loc[price_panel["price_date"] == d, "ticker"])
    missing = sorted(set(all_tickers) - present)
    for t in missing:
        missing_records.append({"price_date": d, "missing_ticker": t})

missing_df = pd.DataFrame(missing_records)

if len(missing_df) > 0:
    print("Missing ticker frequency on dropped days:")
    print(missing_df["missing_ticker"].value_counts())
else:
    print("No missing tickers found on dropped days.")

# 7) Compare sentiment availability vs price availability
print("\n==== SENTIMENT VS PRICE COVERAGE ====")

sent_panel = (
    df[["ticker", "price_date", "sentiment_score"]]
      .copy()
      .dropna(subset=["ticker", "price_date"])
      .groupby(["ticker", "price_date"], as_index=False)
      .agg(
          n_articles=("sentiment_score", "size"),
          sentiment_nonnull=("sentiment_score", lambda x: x.notna().sum()),
      )
)

print("Unique ticker-days with any sentiment rows:",
      sent_panel[["ticker", "price_date"]].drop_duplicates().shape[0])

merged_cov = (
    sent_panel[["ticker", "price_date"]]
    .drop_duplicates()
    .merge(
        price_panel[["ticker", "price_date"]].drop_duplicates().assign(has_price=1),
        on=["ticker", "price_date"],
        how="left"
    )
)

merged_cov["has_price"] = merged_cov["has_price"].fillna(0)

print("\nSentiment ticker-days that also have valid prices:")
print(merged_cov["has_price"].value_counts())

# 8) Final concise diagnosis
print("\n==== DIAGNOSIS SUMMARY ====")
print(f"Raw unique ticker-days: {df[['ticker','price_date']].drop_duplicates().shape[0]}")
print(f"Ticker-days with valid open/close: {price_panel[['ticker','price_date']].drop_duplicates().shape[0]}")
print(f"Days with >= {MIN_TICKERS} tickers: {(ticker_counts['n_tickers'] >= MIN_TICKERS).sum()}")

if (ticker_counts["n_tickers"] >= MIN_TICKERS).sum() < len(ticker_counts):
    print("\nMain sparsity source appears to be: insufficient ticker coverage in valid return data.")
else:
    print("\nMain sparsity source does not appear to be the MIN_TICKERS coverage filter.")

==== RAW DATA CHECKS ====
Total article rows: 89405
Unique tickers: 7
Raw date range: 2024-02-09 00:00:00+00:00 -> 2026-02-24 00:00:00+00:00
Unique ticker-days in raw joined data: 1401

Missingness in price fields:
next_open: 0.00%
next_close: 0.00%

==== PRICE PANEL CHECKS ====
Ticker-day rows after requiring valid open/close: 1401
Unique ticker-days in price panel: 1401
Price panel date range: 2024-02-09 00:00:00+00:00 -> 2026-02-24 00:00:00+00:00

Ticker-day counts by ticker (price panel):
ticker
TSLA     413
AMZN     253
AAPL     187
GOOGL    142
NVDA     142
META     140
MSFT     124
dtype: int64

==== TICKER COVERAGE PER DAY ====
Days with any valid returns: 445

Distribution of ticker coverage per day:
n_tickers
1    183
2     67
3     55
4     17
5      4
6      2
7    117
Name: count, dtype: int64

Summary stats of daily ticker coverage:
count    445.000000
mean       3.148315
std        2.471555
min        1.000000
25%        1.000000
50%        2.000000
75%        7.000000
m

### Camparison Table for Returns

In [27]:
# ----------------------------------------------------
# Lag comparison table: sentiment(t) vs targets(t+1, t+2, t+3)
# ----------------------------------------------------

import pandas as pd

# Start from the daily correlation dataset already built above
lag_df = corr_df[[
    "price_date",
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean"
]].copy()

lag_df = lag_df.sort_values("price_date").reset_index(drop=True)

# Create future targets
for lag in [1, 2, 3]:
    lag_df[f"ret_cs_std_tplus{lag}"] = lag_df["ret_cs_std"].shift(-lag)
    lag_df[f"ret_cs_mad_tplus{lag}"] = lag_df["ret_cs_mad"].shift(-lag)
    lag_df[f"ret_mean_tplus{lag}"] = lag_df["ret_mean"].shift(-lag)

results = []

targets = {
    "ret_cs_std": "ret_cs_std",
    "ret_cs_mad": "ret_cs_mad",
    "ret_mean": "ret_mean"
}

for lag in [1, 2, 3]:
    for target_name, base_col in targets.items():
        future_col = f"{base_col}_tplus{lag}"
        temp = lag_df[["sentiment_score", future_col]].dropna()

        results.append({
            "target": target_name,
            "lag": f"t+{lag}",
            "n_obs": len(temp),
            "pearson_corr": temp["sentiment_score"].corr(temp[future_col], method="pearson"),
            "spearman_corr": temp["sentiment_score"].corr(temp[future_col], method="spearman"),
        })

results_df = pd.DataFrame(results)

print("\nLag comparison table:\n")
print(results_df)

print("\nPivoted view (Pearson):\n")
print(results_df.pivot(index="target", columns="lag", values="pearson_corr"))

print("\nPivoted view (Spearman):\n")
print(results_df.pivot(index="target", columns="lag", values="spearman_corr"))


Lag comparison table:

       target  lag  n_obs  pearson_corr  spearman_corr
0  ret_cs_std  t+1    119     -0.076085       0.019456
1  ret_cs_mad  t+1    119     -0.064748       0.057499
2    ret_mean  t+1    119     -0.204932      -0.103454
3  ret_cs_std  t+2    118     -0.007095       0.044230
4  ret_cs_mad  t+2    118     -0.000548       0.054974
5    ret_mean  t+2    118     -0.035392       0.032251
6  ret_cs_std  t+3    117      0.040059       0.118899
7  ret_cs_mad  t+3    117      0.063823       0.139520
8    ret_mean  t+3    117      0.007821      -0.026076

Pivoted view (Pearson):

lag              t+1       t+2       t+3
target                                  
ret_cs_mad -0.064748 -0.000548  0.063823
ret_cs_std -0.076085 -0.007095  0.040059
ret_mean   -0.204932 -0.035392  0.007821

Pivoted view (Spearman):

lag              t+1       t+2       t+3
target                                  
ret_cs_mad  0.057499  0.054974  0.139520
ret_cs_std  0.019456  0.044230  0.118899
ret_

### Sentiment Bucket Analysis 

In [29]:
bucket_df = corr_df[[
    "price_date",
    "sentiment_score",
    "ret_mean"
]].copy()

bucket_df = bucket_df.sort_values("price_date").reset_index(drop=True)

# Create t+1 returns
bucket_df["ret_mean_t1"] = bucket_df["ret_mean"].shift(-1)

bucket_df = bucket_df.dropna()

# Create sentiment buckets (quantiles)
bucket_df["sent_bucket"] = pd.qcut(
    bucket_df["sentiment_score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Group and analyze
result = bucket_df.groupby("sent_bucket")["ret_mean_t1"].agg(
    ["mean", "std", "count"]
)

print("\nSentiment bucket -> next-day returns:\n")
print(result)


Sentiment bucket -> next-day returns:

                 mean       std  count
sent_bucket                           
Low          0.003858  0.027950     40
Medium       0.002263  0.010298     39
High        -0.001087  0.009798     40
